In [20]:
import json
import numpy as np
import pandas as pd

# 目标坐标 (南极半岛 -63°24'00.0"S, -56°59'00.0"W)
target_lat = -63.40000
target_lon = -56.98333

# ======== 扩大搜索范围 - 包括阿根廷南部 + 南极半岛 ========
# 阿根廷南部: 南纬 40-55度
# 南极半岛: 南纬 60-65度
# 总范围: 南纬 40-70度
tolerance_lat_min = -55.0  # 北界 (阿根廷南部)
tolerance_lat_max = -70.0  # 南界 (南极半岛)
tolerance_lon_min = -80.0  # 西界
tolerance_lon_max = -50.0   # 东界

# 读取 meta_info.json
with open('./meta_info.json', 'r') as f:
    meta_info = json.load(f)

# 搜索附近的气象站
matching_stations = []
for filename, info in meta_info.items():
    lat = info.get('latitude')
    lon = info.get('longitude')
    if lat is None or lon is None:
        continue
    
    # 南纬 40-70度，西经 50-80度
    if -70 <= lat <= -40 and -80 <= lon <= -50:
        matching_stations.append({
            'filename': filename,
            'latitude': lat,
            'longitude': lon,
            'elevation': info.get('ELEVATION'),
            'valid_percent': info.get('valid_percent')
        })

# 按有效率排序
matching_stations = sorted(matching_stations, key=lambda x: x['valid_percent'], reverse=True)

print(f'=== 搜索范围: 南纬 40-70°, 西经 50-80° ===')
print(f'找到 {len(matching_stations)} 个气象站')

# 筛选高质量站点
high_quality = [s for s in matching_stations if s['valid_percent'] > 0.90]
print(f'有效率>90%: {len(high_quality)} 个')

# 显示前30个
print('\n=== 前30个气象站 ===')
for i, station in enumerate(matching_stations[:30]):
    print(f"{i+1}. {station['filename']}")
    print(f"   纬度: {station['latitude']}°N, 经度: {station['longitude']}°W")
    print(f"   海拔: {station['elevation']}m, 有效率: {station['valid_percent']*100:.2f}%")

# 保存CSV
df = pd.DataFrame(high_quality)
df.to_csv('antarctic_stations.csv', index=False)
print(f"\n已保存 {len(high_quality)} 个站点到 antarctic_stations.csv")

=== 搜索范围: 南纬 40-70°, 西经 50-80° ===
找到 22 个气象站
有效率>90%: 22 个

=== 前30个气象站 ===
1. 87938099999.csv
   纬度: -54.843278°N, 经度: -68.29575°W
   海拔: 21.64m, 有效率: 99.86%
2. 87860099999.csv
   纬度: -45.785347°N, 经度: -67.465508°W
   海拔: 57.91m, 有效率: 99.86%
3. 87828099999.csv
   纬度: -43.2105°N, 经度: -65.270319°W
   海拔: 42.97m, 有效率: 99.71%
4. 87925099999.csv
   纬度: -51.608875°N, 经度: -69.312636°W
   海拔: 18.59m, 有效率: 99.68%
5. 87904099999.csv
   纬度: -50.2666666°N, 经度: -72.05°W
   海拔: 199.0m, 有效率: 99.67%
6. 87791099999.csv
   纬度: -40.869222°N, 经度: -63.000389°W
   海拔: 6.09m, 有效率: 99.61%
7. 87909099999.csv
   纬度: -49.306775°N, 经度: -67.802589°W
   海拔: 57.91m, 有效率: 99.50%
8. 89055099999.csv
   纬度: -64.2333333°N, 经度: -56.7166666°W
   海拔: 208.0m, 有效率: 99.32%
9. 89053099999.csv
   纬度: -62.2333333°N, 经度: -58.6333333°W
   海拔: 11.0m, 有效率: 99.31%
10. 87803099999.csv
   纬度: -42.90795°N, 经度: -71.139472°W
   海拔: 798.88m, 有效率: 99.15%
11. 88963099999.csv
   纬度: -63.4°N, 经度: -56.9833333°W
   海拔: 24.0m, 有效率: 98.83%
12. 85

In [21]:
# 提取22个站点的数据，合并为CSV文件
import pandas as pd
import os

data_dir = './global_weather_stations/'
output_path = './antarctic_weather.csv'

# 读取站点列表
df_stations = pd.read_csv('./antarctic_stations.csv')
print(f"共 {len(df_stations)} 个站点")

# 读取第一个站点获取日期范围
first_file = os.path.join(data_dir, df_stations.iloc[0]['filename'])
df_first = pd.read_csv(first_file)
df_first['DATE'] = pd.to_datetime(df_first['DATE'])

# 获取日期范围
start_date = df_first['DATE'].min()
end_date = df_first['DATE'].max()
print(f"日期范围: {start_date} ~ {end_date}")

# 创建时间索引（每小时）- 注意小写h
date_index = pd.date_range(start=start_date, end=end_date, freq='h')
print(f"共 {len(date_index)} 个时间点")

# 准备存储
all_data = {'date': date_index}

# 读取每个站点的气温数据
valid_count = 0
for idx, row in df_stations.iterrows():
    filename = row['filename']
    filepath = os.path.join(data_dir, filename)
    
    if not os.path.exists(filepath):
        print(f"❌ {filename} 不存在")
        continue
    
    try:
        df = pd.read_csv(filepath)
        df['DATE'] = pd.to_datetime(df['DATE'])
        df = df.set_index('DATE')
        
        # 提取气温TMP列
        if 'TMP' in df.columns:
            station_name = filename.replace('.csv', '')
            all_data[station_name] = df['TMP'].reindex(date_index)
            valid_count += 1
    except Exception as e:
        print(f"❌ {filename}: {e}")

print(f"\n成功读取 {valid_count} 个站点")

# 创建数据框
df_weather = pd.DataFrame(all_data)
df_weather.to_csv(output_path, index=False)
print(f"\n已保存到 {output_path}")
print(f"数据形状: {df_weather.shape}")
print(f"\n前几行:\n{df_weather.head()}")

共 22 个站点
日期范围: 2014-01-01 00:00:00 ~ 2023-12-31 23:00:00
共 87648 个时间点

成功读取 22 个站点

已保存到 ./antarctic_weather.csv
数据形状: (87648, 23)

前几行:
                                   date  87938099999  87860099999  \
2014-01-01 00:00:00 2014-01-01 00:00:00          7.2         16.2   
2014-01-01 01:00:00 2014-01-01 01:00:00          6.0         16.0   
2014-01-01 02:00:00 2014-01-01 02:00:00          6.0         15.0   
2014-01-01 03:00:00 2014-01-01 03:00:00          5.5         15.0   
2014-01-01 04:00:00 2014-01-01 04:00:00          5.0         14.0   

                     87828099999  87925099999  87904099999  87791099999  \
2014-01-01 00:00:00    18.274501          7.0      11.8000      20.7000   
2014-01-01 01:00:00    15.397455          7.0      10.1667      20.0111   
2014-01-01 02:00:00    16.691268          6.0       8.5333      19.3222   
2014-01-01 03:00:00    15.200000          5.9       6.9000      18.6333   
2014-01-01 04:00:00    14.800000          6.0       5.5333      17.9444  